In [1]:
# Imports
import pandas as pd
import numpy as np
from tabulate import tabulate
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, BayesianRidge, PoissonRegressor
from difflib import SequenceMatcher
import warnings

warnings.filterwarnings('ignore')
pd.option_context('display.max_rows', None, 'display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)

## Utility Functions

In [2]:
def load_node_profile(node):
    filepath = f'../../tool/infrastructure-profiler/profiles/{node}.csv'

    with open(filepath) as f:
        data = f.readlines()[1].split(',')

    return {
        'ram': int(data[1].strip()),
        'cores': int(data[2].strip()),
        'io_read': float(data[3].strip()),
        'io_write': float(data[4].strip()),
        'sysbench': float(data[5].strip()),
        'z7b': float(data[6].strip()),
        'bench_time': float(data[7].strip())
    }

def combine_df(input_df, trace_df):
    rows = []

    for name, group in input_df.groupby('name'):
        no_files = str(len(group))
        filepaths = group['path']
        paths = ','.join([filepath.split('/')[-1].split('.')[-1] for filepath in filepaths])
        total_size = group['size'].sum() / 1073741824  # convert from B to GiB

        if len(trace_df[trace_df['name'] == name]) > 0:
            task_record = trace_df[trace_df['name'] == name].iloc[0]
            # short_name = task_record['name'].split(':')[-1].split(' ')[0]
            short_name = task_record['name'].split(':')[-2] + ':' + task_record['name'].split(':')[-1].split(' ')[0]
            row = [task_record['task_id'], task_record['name'], task_record['%cpu'], 
                task_record['start'], task_record['complete'], task_record['complete'] - task_record['start'], 
                no_files, paths, total_size, short_name, task_record['hostname']]
            rows.append(row)
        else:
            print('err:', name)

    combined = pd.DataFrame(rows, columns=['id', 'name', '%cpu', 'start', 'complete', 'runtime', 'no_files', 'files', 'input_gib', 'short_name', 'hostname'])
    return combined

def load_profiling_runs(node, workflows, samples, node_profiles):
    node_profile = node_profiles[node]
    runs = {}

    for workflow, samples in zip(workflows, samples):
        runs[workflow] = {}

        for sample in samples:
            sample_dir = f'{node}/{workflow}/{sample}/'
            input_df = pd.read_csv(sample_dir + 'input.csv', sep=';')
            trace_df = pd.read_csv(sample_dir + 'trace.csv')
            runs[workflow][sample] = combine_df(input_df, trace_df)

    return (node_profile, runs)

def load_workflow_execution(cluster, workflow, iteration):
    execution_dir = f'../{cluster}/{workflow}/{iteration}/'
    input_df = pd.read_csv(execution_dir + 'input.csv', sep=';')
    trace_df = pd.read_csv(execution_dir + 'trace.csv')
    return combine_df(input_df, trace_df)

def load_ichnos_trace(cluster, workflow, iteration, gov):
    short_cluster = cluster.split('-')[0]
    trace_loc = f'../{cluster}/{workflow}/ichnos/{short_cluster}-{workflow}-{iteration}-1-{gov}-trace.csv'
    return pd.read_csv(trace_loc)

def process_ichnos_summary(raw):
    core_total_energy = float(raw[7].split(': ')[-1][:-3])
    mem_dynamic_energy = float(raw[9].split(': ')[-1][:-3])
    mem_static_energy = float(raw[15].split(': ')[-1][:-3])
    return {
        'core_total': core_total_energy, 
        'mem_dynamic': mem_dynamic_energy, 
        'mem_static': mem_static_energy, 
        'total': core_total_energy + mem_dynamic_energy + mem_static_energy
    } 

def load_ichnos_summary(cluster, workflow, iteration, gov):
    short_cluster = cluster.split('-')[0]
    with open(f'../{cluster}/{workflow}/ichnos/{short_cluster}-{workflow}-{iteration}-1-{gov}-summary.txt', 'r') as file:
        raw = file.readlines()
    raw = [line.strip() for line in raw]
    return process_ichnos_summary(raw)

def load_rapl_summary(cluster, workflow, iteration):
    rapl_vals = pd.read_csv(f'../{cluster}/{workflow}-runs.csv')
    return rapl_vals.iloc[[iteration - 1]]

def load_power_model(node, setting=''):
    if setting != '':
        filename = f'{setting}-model.txt'
    else: 
        filename = 'model.txt'

    with open(f'../../tool/infrastructure-profiler/raw/{node}/{filename}', 'r') as f:
        data = [line.strip() for line in f.readlines()[1:]]

    coeff = float(data[0].split(',')[0][2:].strip())
    inter = float(data[0].split(',')[1][:-2].strip())
    draw = float(data[1].split(':')[1].strip())

    if draw == 0:
        draw = 0.392

    return (coeff, inter, draw)

def get_task_data_from_experiment(task_name, experiment):
    task_data = experiment[experiment['short_name'] == task_name]
    key_data = task_data[['input_gib', 'runtime', '%cpu']]
    key_data.columns = ['data', 'time', 'usage']
    key_data.reset_index().drop('index', axis=1)
    return key_data

def get_error(predicted, actual):
    return abs(actual - predicted) / actual * 100

In [3]:
class MedianModel:
    def __init__(self, median):
        self.median = median
    
    def predict(self, x):
        x = np.asarray(x)
        value = x.shape[0]
        return np.full(value, self.median)

## Setup Data

In [14]:
# Constants
HU = 'hu'
HU_40 = 'hu-c40'
HU_42 = 'hu-c42'
HU_44 = 'hu-c44'
HU_45 = 'hu-c45'
GPG = 'gpg'
GPG_13 = 'gpgnode-13'
GPG_14 = 'gpgnode-14'
GPG_15 = 'gpgnode-15'
GPG_16 = 'gpgnode-16'
GPG_17 = 'gpgnode-17'
GPG_18 = 'gpgnode-18'
GPG_22 = 'gpgnode-22'
AWS = 'aws'
AWS_c5n = 'aws-metal-c5n'
AWS_z1d = 'aws-metal-z1d'
AWS_c5d = 'aws-metal-c5d'
GCP_n2 = 'gcp-n2'
wfs = {
    GPG: ['rnaseq', 'chipseq', 'nanoseq'], 
    HU: ['rnaseq', 'chipseq', 'rangeland', 'sarek', 'nanoseq', 'atacseq']
}
VR = 'VR'
RF = 'RandomForest'
LR = 'LinearRegression'
BR = 'BayesianRidge'
PR = 'PoissonRegressor'
GBR = 'GradientBoostingRegressor'

In [8]:
# Node Profiles + Power Models (todo)
nodes = [GPG_13, GPG_14, GPG_15, GPG_16, GPG_17, GPG_18, GPG_22, HU_40, HU_42, AWS_c5d, HU_44, HU_45]
node_profiles = {}

for node in nodes:
    profile = load_node_profile(node)
    node_profiles[node] = profile

nodes_by_host = {
    'hu-worker-c40': HU_40,
    'hu-worker-c42': HU_42,
    'hu-worker-c44': HU_44,
    'hu-worker-c45': HU_45
}

gov_by_workflow = {
    GPG: {
        "rnaseq": "ondemand_linear",
        "chipseq": "ondemand_linear",
        "nanoseq": "ondemand_linear",
    },
    HU: {
        "rnaseq": "schedutil_linear",
        "chipseq": "schedutil_linear",
        "rangeland": "schedutil_linear",
        "sarek": "schedutil_linear",
        "nanoseq": "performance_linear",
        "atacseq": "performance_linear"
    }
}

In [18]:
# Workflow Executions 
wf_executions = {}

for cluster, cluster_path in zip([GPG, HU], ['gu-cluster', 'hu-cluster']):
    wf_executions[cluster] = {}
    for wf in wfs[cluster]:
        wf_executions[cluster][wf] = {}
        for iteration in range(1, 4):
            wf_executions[cluster][wf][iteration] = {}
            wf_executions[cluster][wf][iteration]['run'] = load_workflow_execution(cluster_path, wf, iteration)

err: NFCORE_SAREK:SAREK:BAM_VARIANT_CALLING_TUMOR_ONLY_ALL:BAM_VARIANT_CALLING_SINGLE_STRELKA:MERGE_STRELKA (SRR11452475)


In [19]:
# Workflow RAPL Total
for cluster, cluster_path in zip([GPG, HU], ['gu-cluster', 'hu-cluster']):
    for wf in wfs[cluster]:
        for iteration in range(1, 4):
            wf_executions[cluster][wf][iteration]['rapl'] = load_rapl_summary(cluster_path, wf, iteration)

In [20]:
# Workflow Ichnos Total
for cluster, cluster_path in zip([GPG, HU], ['gu-cluster', 'hu-cluster']):
    for wf in wfs[cluster]:
        for iteration in range(1, 4):
            wf_executions[cluster][wf][iteration]['ichnos'] = load_ichnos_summary(cluster_path, wf, iteration, gov_by_workflow[cluster][wf])

In [21]:
# Join Workflow Traces with Ichnos Traces 
def add_energy_with_task_id(run, ichnos):
    combined = pd.merge(run, ichnos[['id', 'core_kwh', 'mem_kwh']], on='id').drop_duplicates()
    return combined[['id', 'name', '%cpu', 'start', 'complete', 'runtime', 'no_files', 'files', 'input_gib', 'short_name', 'hostname', 'core_kwh', 'mem_kwh']]

# Workflow Ichnos Total
for cluster, cluster_path in zip([GPG, HU], ['gu-cluster', 'hu-cluster']):
    for wf in wfs[cluster]:
        for iteration in range(1, 4):
            run = wf_executions[cluster][wf][iteration]['run']
            ichnos_trace = load_ichnos_trace(cluster_path, wf, iteration, gov_by_workflow[cluster][wf])
            df_with_energy = add_energy_with_task_id(run, ichnos_trace)
            wf_executions[cluster][wf][iteration]['run'] = df_with_energy

In [22]:
# Compare RAPL Total to Ichnos Total
gu_socket = {'rnaseq': '_2', 'chipseq': '_2', 'nanoseq': 'both'}
hu_socket = {'rnaseq': '', 'chipseq': '', 'nanoseq': '', 'rangeland': '', 'sarek': '', 'atacseq': ''}

for cluster, cluster_path, socket in zip([GPG, HU], ['gu-cluster', 'hu-cluster'], [gu_socket, hu_socket]):
    print('===', cluster, '===')
    for wf in wfs[cluster]:
        all_ichnos = []
        all_rapl = []

        for iteration in range(1, 4):
            ichnos_total = wf_executions[cluster][wf][iteration]['ichnos']['total']
            ichnos_core = wf_executions[cluster][wf][iteration]['ichnos']['core_total']
            ichnos_mem = wf_executions[cluster][wf][iteration]['ichnos']['mem_dynamic'] + wf_executions[cluster][wf][iteration]['ichnos']['mem_static']

            rapl_socket = socket[wf]
            if rapl_socket == 'both':
                rapl_core = wf_executions[cluster][wf][iteration]['rapl'][f'active_pkg_1'].iloc[0] + wf_executions[cluster][wf][iteration]['rapl'][f'active_pkg_2'].iloc[0]

                if wf_executions[cluster][wf][iteration]['rapl'][f'idle_dram_1'].iloc[0] + wf_executions[cluster][wf][iteration]['rapl'][f'idle_dram_2'].iloc[0] > wf_executions[cluster][wf][iteration]['rapl'][f'dram_1'].iloc[0] + wf_executions[cluster][wf][iteration]['rapl'][f'dram_2'].iloc[0]:
                    rapl_mem = wf_executions[cluster][wf][iteration]['rapl'][f'dram_1'].iloc[0] + wf_executions[cluster][wf][iteration]['rapl'][f'dram_2'].iloc[0]
                else:
                    rapl_mem = wf_executions[cluster][wf][iteration]['rapl'][f'active_dram_1'].iloc[0] + wf_executions[cluster][wf][iteration]['rapl'][f'active_dram_2'].iloc[0]
            else:
                rapl_core = wf_executions[cluster][wf][iteration]['rapl'][f'active_pkg{rapl_socket}'].iloc[0]

                if wf_executions[cluster][wf][iteration]['rapl'][f'idle_dram{rapl_socket}'].iloc[0] > wf_executions[cluster][wf][iteration]['rapl'][f'dram{rapl_socket}'].iloc[0]:
                    rapl_mem = wf_executions[cluster][wf][iteration]['rapl'][f'dram{rapl_socket}'].iloc[0]
                else:
                    rapl_mem = wf_executions[cluster][wf][iteration]['rapl'][f'active_dram{rapl_socket}'].iloc[0]

            rapl_total = rapl_core + rapl_mem
            wf_executions[cluster][wf][iteration]['rapl']['core'] = rapl_core
            wf_executions[cluster][wf][iteration]['rapl']['mem'] = rapl_mem
            core_error = abs(rapl_core - ichnos_core) / rapl_core * 100
            mem_error = abs(rapl_mem - ichnos_mem) / rapl_mem * 100
            total_error = abs(rapl_total - ichnos_total) / rapl_total * 100

            #print(f'{wf}-{iteration} ichnos: {ichnos_total:.2f}kWh rapl: {rapl_total:.2f}kWh error {total_error:.2f}%')
            all_ichnos.append(ichnos_total)
            all_rapl.append(rapl_total)
        avg_ichnos = np.mean(all_ichnos)
        avg_rapl = np.mean(all_rapl)

        print(f'{wf} i: {avg_ichnos:.2f}kWh, r: {avg_rapl:.2f}kWh, {get_error(avg_ichnos, avg_rapl):.2f}%')

=== gpg ===
rnaseq i: 1.52kWh, r: 1.43kWh, 6.99%
chipseq i: 2.00kWh, r: 1.90kWh, 5.64%
nanoseq i: 0.88kWh, r: 0.98kWh, 10.56%
=== hu ===
rnaseq i: 2.28kWh, r: 2.54kWh, 10.20%
chipseq i: 4.68kWh, r: 5.16kWh, 9.22%
rangeland i: 2.16kWh, r: 2.46kWh, 12.10%
sarek i: 4.83kWh, r: 5.28kWh, 8.53%
nanoseq i: 0.42kWh, r: 0.51kWh, 16.31%
atacseq i: 0.61kWh, r: 0.67kWh, 9.95%


In [ ]:
# Load workflow profiling runs
wf_profiles = {}

# HU_40
wfs = ['rnaseq', 'chipseq', 'nanoseq']
samples = [['sample_a', 'sample_b', 'sample_c', 'sample_d', 'sample_e'], 
           ['sample_f', 'sample_g', 'sample_h', 'sample_i', 'sample_j'],
           ['sample_k', 'sample_l', 'sample_m', 'sample_n', 'sample_o']]
profile, runs = load_profiling_runs(HU_40, wfs, samples, node_profiles)
wf_profiles[HU_40] = {
    'node_profile': profile,
    'executions': runs,
    'power_model': load_power_model(HU_40, 'schedutil')
}

# HU_42
wfs = ['rnaseq', 'chipseq']
samples = [['sample_a', 'sample_b', 'sample_c', 'sample_d', 'sample_e'], ['sample_g', 'sample_h', 'sample_i', 'sample_j']]  # sample_f to be added back
profile, runs = load_profiling_runs(HU_42, wfs, samples, node_profiles)
wf_profiles[HU_42] = {
    'node_profile': profile,
    'executions': runs,
    'power_model': load_power_model(HU_42, 'schedutil')
}

HU_44
wfs = ['rnaseq', 'chipseq']
samples = [['sample_a', 'sample_b', 'sample_c', 'sample_d', 'sample_e'], ['sample_f', 'sample_g', 'sample_h', 'sample_i', 'sample_j']]  
profile, runs = load_profiling_runs(HU_44, wfs, samples, node_profiles)
wf_profiles[HU_44] = {
    'node_profile': profile,
    'executions': runs,
    'power_model': load_power_model(HU_44, 'schedutil')
}

HU_45
wfs = ['rnaseq', 'chipseq']
samples = [['sample_a', 'sample_b', 'sample_c', 'sample_d', 'sample_e'], ['sample_f', 'sample_g', 'sample_h', 'sample_i', 'sample_j']]  
profile, runs = load_profiling_runs(HU_45, wfs, samples, node_profiles)
wf_profiles[HU_45] = {
    'node_profile': profile,
    'executions': runs,
    'power_model': load_power_model(HU_45, 'schedutil')
}

# gpgnode-13
wf_profiles[GPG_13] = {
    'node_profile': node_profiles[GPG_13],
    'power_model': load_power_model(GPG_13, 'ondemand')
}

# gpgnode-14
wf_profiles[GPG_14] = {
    'node_profile': node_profiles[GPG_14],
    'power_model': load_power_model(GPG_14, 'ondemand')
}

# gpgnode-15
wf_profiles[GPG_15] = {
    'node_profile': node_profiles[GPG_15],
    'power_model': load_power_model(GPG_15, 'ondemand')
}

# gpgnode-16
wf_profiles[GPG_16] = {
    'node_profile': node_profiles[GPG_16],
    'power_model': load_power_model(GPG_16, 'ondemand')
}

# gpgnode-17
wf_profiles[GPG_17] = {
    'node_profile': node_profiles[GPG_17],
    'power_model': load_power_model(GPG_17, 'ondemand')
}

# gpgnode-18
wfs = ['rnaseq', 'chipseq', 'nanoseq']
samples = [['sample_a', 'sample_b', 'sample_c', 'sample_d', 'sample_e'], 
           ['sample_f', 'sample_g', 'sample_h', 'sample_i', 'sample_j'], 
           ['sample_k', 'sample_l', 'sample_m', 'sample_n', 'sample_o']]
profile, runs = load_profiling_runs(GPG_18, wfs, samples, node_profiles)
wf_profiles[GPG_18] = {
    'node_profile': node_profiles[GPG_18],
    'executions': runs,
    'power_model': load_power_model(GPG_18, 'ondemand')
}

# AWS metal c5d
wfs = ['rnaseq', 'chipseq']
samples = [['sample_a', 'sample_b', 'sample_c', 'sample_d', 'sample_e'], ['sample_g', 'sample_h', 'sample_i', 'sample_j']]  # sample_f to be added back
profile, runs = load_profiling_runs(AWS_c5d, wfs, samples, node_profiles)
wf_profiles[AWS_c5d] = {
    'node_profile': node_profiles[AWS_c5d],
    'executions': runs,
    'power_model': load_power_model(AWS_c5d)
}

err: NFCORE_CHIPSEQ:CHIPSEQ:IGV
err: NFCORE_CHIPSEQ:CHIPSEQ:IGV


In [ ]:
# @Vasilis
# profiling data -- mini executions using downsampled data on a single node
for node in wf_profiles.keys():
    if 'executions' in wf_profiles[node]:
        print(wf_profiles[node]['executions']['rnaseq'])
        # print(wf_profiles[node]['executions']['chipseq'])
        # print(wf_profiles[node]['executions']['nanoseq'])

# workflow data -- full executions on a cluster 
for cluster, workflows in zip([HU, GPG], [['rnaseq', 'chipseq', 'nanoseq'], ['rnaseq', 'chipseq', 'nanoseq']]):
    for workflow in workflows:
        print(wf_executions[cluster][workflow][1]['run']) # run 1, 2 and 3 

{'sample_a':     id                                               name   %cpu  \
0    2  NFCORE_RNASEQ:PREPARE_GENOME:CUSTOM_GETCHROMSI...   99.9   
1    1  NFCORE_RNASEQ:PREPARE_GENOME:GTF_FILTER (genom...   99.8   
2    6  NFCORE_RNASEQ:PREPARE_GENOME:MAKE_TRANSCRIPTS_...  100.1   
3    7  NFCORE_RNASEQ:PREPARE_GENOME:SALMON_INDEX (gen...  425.5   
4   16  NFCORE_RNASEQ:RNASEQ:ALIGN_STAR:BAM_SORT_STATS...  135.8   
5   14  NFCORE_RNASEQ:RNASEQ:ALIGN_STAR:BAM_SORT_STATS...  123.1   
6   15  NFCORE_RNASEQ:RNASEQ:ALIGN_STAR:BAM_SORT_STATS...  120.3   
7   12  NFCORE_RNASEQ:RNASEQ:ALIGN_STAR:BAM_SORT_STATS...  142.2   
8   10  NFCORE_RNASEQ:RNASEQ:ALIGN_STAR:BAM_SORT_STATS...  563.6   
9    9  NFCORE_RNASEQ:RNASEQ:ALIGN_STAR:STAR_ALIGN_IGE...  274.3   
10  34  NFCORE_RNASEQ:RNASEQ:BAM_MARKDUPLICATES_PICARD...  132.2   
11  32  NFCORE_RNASEQ:RNASEQ:BAM_MARKDUPLICATES_PICARD...   95.0   
12  33  NFCORE_RNASEQ:RNASEQ:BAM_MARKDUPLICATES_PICARD...  119.6   
13  13  NFCORE_RNASEQ:RNASEQ:BAM_MA

## Evaluation

In [ ]:
# Augur Functions
def custom_fallback(task_name):
    #print(f"No match found for {task_name}, using default config") # todo @Kath !!!
    return ''

def find_closest_key(task_name, dictionary):
    task_parts = task_name.split('_')
    best_match = None
    best_score = 0

    for key in list(dictionary.keys()):
        key_parts = key.split('_')
        current_score = 0

        for pos, task_part in enumerate(task_parts):
            if task_part in key_parts:
                key_position = key_parts.index(task_part)
                position_weight = 1.0 / (pos + 1) + 1.0 / (key_position + 1)
                current_score += position_weight

        if task_parts[0] == key_parts[0]:
            current_score += 2.0  # first word importance        

        if current_score > best_score:
            best_score = current_score
            best_match = key 

    if best_match: 
        return best_match
    else: 
        return custom_fallback(task_name)

def compute_active_time_per_host(execution):
    tasks_by_host = {}

    for _, record in execution.iterrows():
        host = record['hostname']
        start_ts = record['start']
        end_ts = record['complete']

        if host in tasks_by_host:
            tasks_by_host[host].append((start_ts, end_ts))
        else:
            tasks_by_host[host] = [(start_ts, end_ts)]

    active_time_by_host = {}

    for host, intervals in tasks_by_host.items():
        if not intervals:
            continue

        intervals.sort(key=lambda x: x[0])  # sort by start timestamp
        merged = []
        current_start, current_end = intervals[0]

        for start, end in intervals[1:]:
            if start <= current_end:  # overlap or contiguous
                current_end = max(current_end, end)
            else:
                merged.append((current_start, current_end))
                current_start, current_end = start, end
        merged.append((current_start, current_end))

        total_ms: float = sum(end - start for start, end in merged)
        active_time_by_host[host] = total_ms / 1000 / 3600  # return time in hours

    return active_time_by_host

def get_runtime_translation_factor(raw_factor, local_profile, target_profile):
    return (0.5 * raw_factor) + (0.5 * (local_profile['cores'] / target_profile['cores']))
    # scale runtime and consider how the difference in target cores impacts across machines

def get_translation_factor(local_profile, target_profile):
    local_write_val = local_profile['io_write'] / local_profile['ram']
    local_read_val = local_profile['io_read'] / local_profile['ram']
    target_write_val = target_profile['io_write'] / target_profile['ram']
    target_read_val = target_profile['io_read'] / target_profile['ram']
    return \
        (0.20 * local_profile['z7b'] / target_profile['z7b']) + \
        (0.20 * local_profile['sysbench'] / target_profile['sysbench']) + \
        (0.25 * local_write_val / target_write_val) + \
        (0.25 * local_read_val / target_read_val) + \
        (0.10 * local_profile['bench_time'] / target_profile['bench_time'])

In [ ]:
# @Vasilis -- latest version of prediction model combining profiling runs with workflow executions (of other workflows)
def get_task_data_from_cluster_runs(runs, exclude_workflow):
    all_task_data = {}

    for workflow in runs.keys():
        if workflow == exclude_workflow:
            continue

        iterations = range(1, 4)
        tasks = runs[workflow][iterations[0]]['run']['short_name'].unique()

        for task in tasks:
            task_data = pd.DataFrame(columns=['data', 'time', 'usage'], dtype='float64')
            for iteration in iterations:
                experiment_data = get_task_data_from_experiment(task, runs[workflow][iteration]['run'])
                task_data = pd.concat([task_data, experiment_data])

            all_task_data[task] = task_data

    return all_task_data

def variance_clumping_score(x):
    val = np.var(x)
    return 1.0 / val if val != 0 else np.inf

def create_prediction_model_with_reflections(profiling_runs, model_type, cluster_runs, exclude_workflow):
    models = {}
    reflector_data = get_task_data_from_cluster_runs(cluster_runs, exclude_workflow)

    for workflow in profiling_runs.keys():
        models[workflow] = {}
        samples = list(profiling_runs[workflow].keys()) 
        tasks = profiling_runs[workflow][samples[0]]['short_name'].unique() 

        for task in tasks:
            task_data = pd.DataFrame(columns=['data', 'time', 'usage'], dtype='float64')
            for sample in samples: 
                experiment_data = get_task_data_from_experiment(task, profiling_runs[workflow][sample])
                task_data = pd.concat([task_data, experiment_data])

            models[workflow][task] = {}
            models[workflow][task]['usage'] = np.mean(task_data['usage'])

            # @Vasilis where refelctor data is added to the task data
            if task in reflector_data.keys():
                task_data = pd.concat([task_data, reflector_data[task]])

            # identify tasks with data with low variance / other @Vasilis 
            # models[workflow][task]['variance'] = np.std(task_data['data'])
            models[workflow][task]['variance'] = variance_clumping_score(task_data['data'])
            pearson = np.corrcoef(task_data['data'].to_numpy(), task_data['time'].to_numpy())[0, 1]

            if np.isnan(pearson) or pearson < 0.75:  
               models[workflow][task]['model'] = MedianModel(np.median(task_data['time'].to_numpy())) 
               models[workflow][task]['model_type'] = 'MedianModel'
            # second case
            else:
                if model_type == BR:
                    model = BayesianRidge(fit_intercept=True)
                    models[workflow][task]['model_type'] = 'BayesianRidge'
                elif model_type == GBR:
                    model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3)
                    models[workflow][task]['model_type'] = 'GBR'
                elif model_type == VR:
                    model = SVR()
                    models[workflow][task]['model_type'] = 'VectorRegression'
                elif model_type == RF:
                    model = RandomForestRegressor()
                    models[workflow][task]['model_type'] = 'RandomForestRegressor'
                elif model_type == PR:
                    model = PoissonRegressor(fit_intercept=True)
                    models[workflow][task]['model_type'] = 'PoissonRegressor'
                else:
                    model = LinearRegression()
                    models[workflow][task]['model_type'] = 'LinearRegression'

                models[workflow][task]['model'] = model.fit(task_data['data'].values.reshape(-1, 1), task_data['time'].values) 
                if task in ['BAM_RSEQC:RSEQC_JUNCTIONSATURATION', 'RNASEQ:QUALIMAP_RNASEQ']:
                    print(task)
                    print(task_data)
                    print('coef', model.coef_)
                    print('intercept', model.intercept_)

    return models

In [ ]:
# Prediction and Verification
headers = ['task', 'in (GiB)', 'pred_t (s)', 'scaled_pred_t', 'actual_t', 'raw_error', 'scaled_error', 
           'pred_u (%)', 'actual_u', 'pred_cpu', 'actual_cpu', 'pred_mem', 'actual_mem', 'node', 'files', 'no_files', 'variance', 'type']

def predict_wf(wf_model, workflow, local_profile): 
    data = []
    overall_error = []
    energy_error = []
    raw_error_t = []
    abs_raw_diff_t = []
    abs_scaled_diff_t = []
    missing_tasks = [] 
    actual_execution = workflow['run']
    total_cpu_dynamic_energy = 0.0
    total_mem_dynamic_energy = 0.0

    for _, record in actual_execution.iterrows():
        task_name = record['short_name']
        data_size = record['input_gib']
        node = record['hostname']
        actual_runtime = record['runtime']
        actual_usage = record["%cpu"]

        if actual_runtime > 5000:  # exclude tasks that run for 0-5 seconds
            prediction_model = None

            if task_name not in wf_model.keys():
                missing_tasks.append((task_name))
                continue
            else:
                model_name = task_name

            if prediction_model is None:
                prediction_model = wf_model[model_name]['model']
                pred_usage = wf_model[model_name]['usage']

            if node not in wf_profiles.keys():
                node = nodes_by_host[node]

            target_profile = wf_profiles[node]['node_profile']
            factor = get_translation_factor(local_profile, target_profile)
            factor = get_runtime_translation_factor(factor, local_profile, target_profile)
            target_power_model = wf_profiles[node]['power_model']
            predicted_runtime_raw = prediction_model.predict([[data_size]])[0] 
            predicted_runtime = predicted_runtime_raw * factor  

            if predicted_runtime > 0:
                error = (abs(predicted_runtime - actual_runtime) / actual_runtime) * 100 
                overall_error.append(error)
                raw_error = (abs(predicted_runtime_raw - actual_runtime) / actual_runtime) * 100 
                raw_error_t.append(raw_error)
                abs_raw_diff_t.append(abs(predicted_runtime_raw - actual_runtime))
                abs_scaled_diff_t.append(abs(predicted_runtime - actual_runtime))

            # estimate energy consumption (ichnos logic)
            time_h = predicted_runtime / 1000 / 3600  # convert from ms to hours
            memory = data_size
            cpu_usage = pred_usage / target_profile['cores'] # energy calculation logic NOT CPU PREDICTION %
            core_consumption = time_h * (target_power_model[0] * cpu_usage) * 0.001
            mem_consumption = memory * target_power_model[2] * time_h * 0.001
            total_cpu_dynamic_energy += core_consumption
            total_mem_dynamic_energy += mem_consumption
            predicted_energy = core_consumption + mem_consumption
            actual_energy = record['core_kwh'] + record['mem_kwh']

            if actual_energy > 0 and predicted_energy > 0:
                energy_error.append(get_error(predicted_energy, actual_energy))

            # report
            data.append([task_name, data_size, predicted_runtime_raw / 1000, predicted_runtime / 1000, actual_runtime / 1000, raw_error, error, 
                         pred_usage, actual_usage, core_consumption, record['core_kwh'], mem_consumption, record['mem_kwh'], node, record['files'], 
                         record['no_files'], wf_model[model_name]['variance'], wf_model[model_name]['model_type']])

    task_runtime_error = np.mean(overall_error)
    task_energy_error = np.mean(energy_error)
    task_runtime_error_raw = np.mean(raw_error_t)
    print('raw task runtime error%', task_runtime_error_raw)
    print('abs raw diff time', np.sum(abs_raw_diff_t) / 1000 )
    print('abs scaled diff time', np.sum(abs_scaled_diff_t) / 1000) # in seconds

    total_cpu_static_energy = 0.0
    total_mem_static_energy = 0.0
    active_time_per_host = compute_active_time_per_host(actual_execution)
    for host in active_time_per_host.keys():
        total_cpu_static_energy += active_time_per_host[host] * target_power_model[1] * 0.001  # convert to kWh
        total_mem_static_energy += active_time_per_host[host] * target_power_model[2] * target_profile['ram'] * 0.001 # convert to kWh

    augur_total_cpu = total_cpu_dynamic_energy + total_cpu_static_energy
    augur_total_mem = total_mem_dynamic_energy + total_mem_static_energy
    augur_total = augur_total_cpu + augur_total_mem
    ichnos_total = workflow['ichnos']['total']
    rapl_total = float(workflow['rapl']['core']) + float(workflow['rapl']['mem'])
    data_df = pd.DataFrame(data, columns=headers)

    return ((augur_total_cpu, augur_total_mem, augur_total), ichnos_total, rapl_total, (task_runtime_error, task_energy_error, data, missing_tasks, data_df))

In [ ]:
# @Vasilis
print(wf_profiles[HU_40]['executions']['rnaseq']['sample_a'])

    id                                               name   %cpu  \
0    2  NFCORE_RNASEQ:PREPARE_GENOME:CUSTOM_GETCHROMSI...   99.9   
1    1  NFCORE_RNASEQ:PREPARE_GENOME:GTF_FILTER (genom...   99.8   
2    6  NFCORE_RNASEQ:PREPARE_GENOME:MAKE_TRANSCRIPTS_...  100.1   
3    7  NFCORE_RNASEQ:PREPARE_GENOME:SALMON_INDEX (gen...  425.5   
4   16  NFCORE_RNASEQ:RNASEQ:ALIGN_STAR:BAM_SORT_STATS...  135.8   
5   14  NFCORE_RNASEQ:RNASEQ:ALIGN_STAR:BAM_SORT_STATS...  123.1   
6   15  NFCORE_RNASEQ:RNASEQ:ALIGN_STAR:BAM_SORT_STATS...  120.3   
7   12  NFCORE_RNASEQ:RNASEQ:ALIGN_STAR:BAM_SORT_STATS...  142.2   
8   10  NFCORE_RNASEQ:RNASEQ:ALIGN_STAR:BAM_SORT_STATS...  563.6   
9    9  NFCORE_RNASEQ:RNASEQ:ALIGN_STAR:STAR_ALIGN_IGE...  274.3   
10  34  NFCORE_RNASEQ:RNASEQ:BAM_MARKDUPLICATES_PICARD...  132.2   
11  32  NFCORE_RNASEQ:RNASEQ:BAM_MARKDUPLICATES_PICARD...   95.0   
12  33  NFCORE_RNASEQ:RNASEQ:BAM_MARKDUPLICATES_PICARD...  119.6   
13  13  NFCORE_RNASEQ:RNASEQ:BAM_MARKDUPLICATES_

In [ ]:
# HU to GPG with and without reflector 
# Bayesian / Ridge regression is supposed to mitgiate these wild coefficients/values
# variance low - in absolute terms as well as std 
hu_wf_models = create_prediction_model_with_reflections(wf_profiles[HU_40]['executions'], LR, wf_executions[GPG], 'rnaseq')
local_profile = wf_profiles[HU_40]['node_profile']
hu_rnaseq_model = hu_wf_models['rnaseq']
(_, _, _, ((task_runtime_error, task_energy_error, data, missing_tasks, data_df))) = predict_wf(hu_rnaseq_model, wf_executions[GPG]['rnaseq'][1], local_profile)

print('runtime%', task_runtime_error, 'energy%', task_energy_error)
print('missing', set(missing_tasks))

explore = data_df.groupby(['task', 'type']).sum('raw_error').reset_index()
# print(explore.columns)
print(explore.sort_values(by='raw_error', ascending=False)[['task', 'in (GiB)', 'pred_t (s)', 'scaled_pred_t', 'actual_t',
       'raw_error', 'scaled_error', 'variance', 'type']])

BAM_RSEQC:RSEQC_JUNCTIONSATURATION
        data     time  usage
19  0.193829  15671.0  145.6
19  0.199311  15503.0  147.6
19  0.176177  13949.0  152.5
19  0.154472  13006.0  149.6
19  0.271353  20724.0  133.9
coef [67414.49758809]
intercept 2353.205207735784
RNASEQ:QUALIMAP_RNASEQ
        data      time  usage
37  0.950609  127912.0  136.6
37  0.956091  131394.0  136.9
37  0.932957  116145.0  141.0
37  0.911252  105067.0  140.9
37  1.028133  171644.0  125.3
coef [573139.41926139]
intercept -417379.1070586438
raw task runtime error% 44.90107171312481
abs raw diff time 25544.58193870257
abs scaled diff time 21749.10306195257
runtime% 34.33632184080573 energy% 108.3198859228456
missing {'RNASEQ:SAMTOOLS_SORT_QUALIMAP'}
                                                 task    in (GiB)   pred_t (s)  scaled_pred_t  actual_t   raw_error  scaled_error      variance              type
13                  BAM_STATS_SAMTOOLS:SAMTOOLS_STATS   54.306653   596.374630    1098.548957    1411.0  691.758

In [ ]:
# AWS to GPG
aws_wf_models = create_prediction_model_with_reflections(wf_profiles[AWS_c5d]['executions'], LR, wf_executions[GPG], 'rnaseq')
local_profile = wf_profiles[AWS_c5d]['node_profile']
target_profile = wf_profiles[GPG_16]['node_profile']
target_power_model = wf_profiles[GPG_16]['power_model']
aws_rnaseq_model = aws_wf_models['rnaseq']
(_, _, _, ((task_runtime_error, task_energy_error, data, missing_tasks, data_df))) = predict_wf(aws_rnaseq_model, wf_executions[GPG]['rnaseq'][1], local_profile)

print('runtime', task_runtime_error, 'energy', task_energy_error)

explore = data_df.groupby(['task', 'type']).sum('raw_error').reset_index()
print(explore.sort_values(by='raw_error', ascending=False)[['task', 'in (GiB)', 'pred_t (s)', 'scaled_pred_t', 'actual_t',
      'raw_error', 'scaled_error', 'variance', 'type']])
# print out the data input / what causes this, e.g. 1-2 files, are they even fastq.gz/other? tsv files??? 

raw task runtime error% 75.24154770643551
abs raw diff time 36389.77496851759
abs scaled diff time 24878.816139737188
runtime 72.08854291584483 energy 122.62441539540326
                                                 task    in (GiB)   pred_t (s)  scaled_pred_t  actual_t    raw_error  scaled_error      variance              type
30         QUANTIFY_PSEUDO_ALIGNMENT:TXIMETA_TXIMPORT    0.071604   625.678477    1268.579502      40.0  1464.196191   3071.448755  4.891647e+09  LinearRegression
13                  BAM_STATS_SAMTOOLS:SAMTOOLS_STATS   54.306653   593.455439    1203.512512    1411.0   694.484758    174.834970  1.173037e+01  LinearRegression
43                      RNASEQ:SAMTOOLS_SORT_QUALIMAP   27.441159    41.958000      85.150635    1840.0   585.067884    569.692157  3.864624e+03       MedianModel
9                     BAM_RSEQC:RSEQC_READDUPLICATION    9.854312    83.616000     169.702946    2043.0   574.837108    548.934322  3.864624e+03       MedianModel
3              

In [ ]:
def predict_multiple_executions(local_node, local_profile, workflow, cluster, method):
    wf_models = create_prediction_model_with_reflections(wf_profiles[local_node]['executions'], method, wf_executions[cluster], workflow)
    local_wf_model = wf_models[workflow]
    augur_results = []
    ichnos_results = []
    rapl_results = []
    task_rt_results = []
    task_en_results = []

    for iteration in range(1, 4):
        (augur, ichnos, rapl, task_data) = predict_wf(local_wf_model, wf_executions[cluster][workflow][iteration], local_profile)
        augur_results.append(augur[2])
        ichnos_results.append(ichnos)
        rapl_results.append(rapl)
        task_rt_results.append(task_data[0])
        task_en_results.append(task_data[1])

    augur_avg = np.mean(augur_results)
    ichnos_avg = np.mean(ichnos_results)
    rapl_avg = np.mean(rapl_results)
    avg_rt_error = np.mean(task_rt_results)
    avg_en_error = np.mean(task_en_results)

    return[workflow,local_node,cluster,avg_rt_error,avg_en_error,get_error(augur_avg, ichnos_avg),get_error(augur_avg, rapl_avg)]

def get_all_predictions(model):
    return [
        # SAME TO SAME (HU 40 -> HU, GPG 18 -> GPG)
        predict_multiple_executions(HU_40, wf_profiles[HU_40]['node_profile'], 'rnaseq', HU, model),
        predict_multiple_executions(HU_40, wf_profiles[HU_40]['node_profile'], 'chipseq', HU, model),
        predict_multiple_executions(HU_40, wf_profiles[HU_40]['node_profile'], 'nanoseq', HU, model),
        predict_multiple_executions(GPG_18, wf_profiles[GPG_18]['node_profile'], 'rnaseq', GPG, model),
        predict_multiple_executions(GPG_18, wf_profiles[GPG_18]['node_profile'], 'chipseq', GPG, model),
        predict_multiple_executions(GPG_18, wf_profiles[GPG_18]['node_profile'], 'nanoseq', GPG, model),

        # HU 40
        predict_multiple_executions(HU_40, wf_profiles[HU_40]['node_profile'], 'rnaseq', GPG, model),
        predict_multiple_executions(HU_40, wf_profiles[HU_40]['node_profile'], 'chipseq', GPG, model),
        predict_multiple_executions(HU_40, wf_profiles[HU_40]['node_profile'], 'nanoseq', GPG, model),

        # AWS c5d
        predict_multiple_executions(AWS_c5d, wf_profiles[AWS_c5d]['node_profile'], 'rnaseq', GPG, model),
        predict_multiple_executions(AWS_c5d, wf_profiles[AWS_c5d]['node_profile'], 'chipseq', GPG, model),

        predict_multiple_executions(AWS_c5d, wf_profiles[AWS_c5d]['node_profile'], 'rnaseq', HU, model),
        predict_multiple_executions(AWS_c5d, wf_profiles[AWS_c5d]['node_profile'], 'chipseq', HU, model),

        # GPG 18
        predict_multiple_executions(GPG_18, wf_profiles[GPG_18]['node_profile'], 'rnaseq', HU, model),
        predict_multiple_executions(GPG_18, wf_profiles[GPG_18]['node_profile'], 'chipseq', HU, model),
        predict_multiple_executions(GPG_18, wf_profiles[GPG_18]['node_profile'], 'nanoseq', HU, model),

        # GPG 22
    ]

In [ ]:
# PREDICTION COMPARISONS 
pred_headers = ['workflow', 'local_node', 'target_cluster', 'task_runtime_err', 'task_energy_err', 'wf_ichnos_err', 'wf_rapl_err']

In [ ]:
# augur predictions using a Linear Regression model
model = LR
data_lr = get_all_predictions(model)
print('linear regression')
print(tabulate(data_lr, headers=pred_headers))

BAM_RSEQC:RSEQC_JUNCTIONSATURATION
        data     time  usage
19  0.193829  15671.0  145.6
19  0.199311  15503.0  147.6
19  0.176177  13949.0  152.5
19  0.154472  13006.0  149.6
19  0.271353  20724.0  133.9
coef [67414.49758809]
intercept 2353.205207735784
RNASEQ:QUALIMAP_RNASEQ
        data      time  usage
37  0.950609  127912.0  136.6
37  0.956091  131394.0  136.9
37  0.932957  116145.0  141.0
37  0.911252  105067.0  140.9
37  1.028133  171644.0  125.3
coef [573139.41926139]
intercept -417379.1070586438
raw task runtime error% 21.390071453373764
abs raw diff time 21861.22563573179
abs scaled diff time 56327.276013229784
raw task runtime error% 22.717765422050498
abs raw diff time 22452.873964086935
abs scaled diff time 54289.295507902505
raw task runtime error% 23.1508945815366
abs raw diff time 23204.393300582335
abs scaled diff time 54217.62222858529
BAM_RSEQC:RSEQC_JUNCTIONSATURATION
         data      time  usage
19   0.193829   15671.0  145.6
19   0.199311   15503.0  147.6
19

notes + thoughts
- why is nanoseq memory error so much higher? 
- our ichnos vs. rapl readings have a greater error, nanoseq has one of the smallest footprints/runtimes- ichnos becomes less accurate for these shorter running tasks
  - esp. when we depend on nextflow traces for data
- why is the overall energy prediction for workflow less accurate when our task runtime/energy prediction is better?
  - we are underestimating overall energy
  - we might miss some tasks because of unseen input data even with the reflector, adding some random value isn't meaningful (could have an additional error checking value where we then add in the actual runtime of the unseen task for the dynamic energy prediction, but unsure?) would have some static contribution as well 
- explore the missing tasks available

- there is still room to explore different regression models, using the data we have!

In [ ]:
# augur predictions using a Bayesian Ridge model
model = BR
data = get_all_predictions(model)
print('bayesian ridge')
print(tabulate(data, headers=pred_headers))

BAM_RSEQC:RSEQC_JUNCTIONSATURATION
        data     time  usage
19  0.193829  15671.0  145.6
19  0.199311  15503.0  147.6
19  0.176177  13949.0  152.5
19  0.154472  13006.0  149.6
19  0.271353  20724.0  133.9
coef [7.37286547e-05]
intercept 15770.599985325909
RNASEQ:QUALIMAP_RNASEQ
        data      time  usage
37  0.950609  127912.0  136.6
37  0.956091  131394.0  136.9
37  0.932957  116145.0  141.0
37  0.911252  105067.0  140.9
37  1.028133  171644.0  125.3
coef [8.71610051e-06]
intercept 130432.39999166907
raw task runtime error% 69.85505326662685
abs raw diff time 152899.4463798915
abs scaled diff time 146049.29367108608
raw task runtime error% 71.03802474190076
abs raw diff time 153983.4196543519
abs scaled diff time 147806.07568098724
raw task runtime error% 71.23009420897029
abs raw diff time 153849.92206048386
abs scaled diff time 147256.2805631943
BAM_RSEQC:RSEQC_JUNCTIONSATURATION
         data      time  usage
19   0.193829   15671.0  145.6
19   0.199311   15503.0  147.6
19  

In [ ]:
# augur predictions using a Gradient Boosting Regressor  model
model = GBR
data = get_all_predictions(model)
print('Gradient Boosting Regressor')
print(tabulate(data, headers=pred_headers))

BAM_RSEQC:RSEQC_JUNCTIONSATURATION
        data     time  usage
19  0.193829  15671.0  145.6
19  0.199311  15503.0  147.6
19  0.176177  13949.0  152.5
19  0.154472  13006.0  149.6
19  0.271353  20724.0  133.9


AttributeError: 'GradientBoostingRegressor' object has no attribute 'coef_'

In [ ]:
# augur predictions using a Vector Regression model
model = VR
data = get_all_predictions(model)
print('Vector Regression')
print(tabulate(data, headers=pred_headers))

Vector Regression
workflow    local_node     target_cluster      task_runtime_err    task_energy_err    wf_ichnos_err    wf_rapl_err
----------  -------------  ----------------  ------------------  -----------------  ---------------  -------------
rnaseq      hu-c40         hu                           73.4322            83.0093          30.7922        37.8498
chipseq     hu-c40         hu                           70.6243            83.2559          22.9841        30.0813
nanoseq     hu-c40         hu                           68.2159            77.7038          37.2274        47.468
rnaseq      hu-c40         gpg                          66.598             89.0658          17.4318        11.6636
chipseq     hu-c40         gpg                          65.8125            84.5451          16.972         12.2889
nanoseq     hu-c40         gpg                          84.3841            94.8815          23.4709        31.5528
rnaseq      aws-metal-c5d  gpg                          68.4563

In [ ]:
# augur predictions using a RandomForest model
model = RF
data = get_all_predictions(model)
print('RandomForest')
print(tabulate(data, headers=pred_headers))

RandomForest
workflow    local_node     target_cluster      task_runtime_err    task_energy_err    wf_ichnos_err    wf_rapl_err
----------  -------------  ----------------  ------------------  -----------------  ---------------  -------------
rnaseq      hu-c40         hu                           64.7212            75.4895          30.171         37.292
chipseq     hu-c40         hu                           63.1016            76.0612          22.6223        29.7528
nanoseq     hu-c40         hu                           62.2858            73.0746          36.8142        47.1221
rnaseq      hu-c40         gpg                          64.128             87.9409          17.198         11.4134
chipseq     hu-c40         gpg                          83.5997            82.2749          16.8912        12.2035
nanoseq     hu-c40         gpg                          81.4709            93.3196          23.1268        31.2451
rnaseq      aws-metal-c5d  gpg                          67.7813     

below will be used to generate tables / data ready to plug into the paper